# Using the Saved Pipeline to Make Predictions

This is the "deployment" side of `titanic-using-pipeline.ipynb`: load the single pickled pipeline it exported, and use it to predict on brand-new, raw input — no manual imputing, encoding, or scaling needed, since the pipeline already knows how to do all of that internally.

## 1. Load the Saved Pipeline

### A Quick Security Note on `pickle`
Unpickling a file can execute arbitrary code — only load `.pkl` files you created yourself or trust completely, never one from an untrusted source.

In [1]:
import pickle
import numpy as np

In [2]:
pipe = pickle.load(open('pipe.pkl','rb'))

FileNotFoundError: [Errno 2] No such file or directory: 'pipe.pkl'

## 2. Preparing a Raw Input

The array's 7 values correspond to the *original* raw columns, in the exact order the pipeline was trained on: `Pclass, Sex, Age, SibSp, Parch, Fare, Embarked`. Note there's no encoding or scaling happening here at all — `'male'` and `'S'` are passed in as plain strings, exactly as they'd appear in a real, un-preprocessed row.

In [ ]:
# Pclass, Sex, Age, SibSp, Parch, Fare, Embarked


In [ ]:
# Assume user input
test_input2 = np.array([2, 'male', 31.0, 0, 0, 10.5, 'S'],dtype=object).reshape(1,7)

## 3. Predicting

The pipeline's `.predict()` runs the raw input through *every* fitted step internally — impute, encode, scale, select features, then classify — and returns just the final class: `0` (did not survive) or `1` (survived).

In [ ]:
pipe.predict(test_input2)

### Getting a Probability, Not Just a Label — `predict_proba()`

`.predict()` only reveals the final decision. `.predict_proba()` shows *how confident* the model is — often far more useful, since a 51%-vs-49% call and a 99%-vs-1% call both round to the same predicted class but mean very different things.

In [ ]:
probabilities = pipe.predict_proba(test_input2)
print(f"P(did not survive) = {probabilities[0][0]:.2%}")
print(f"P(survived)        = {probabilities[0][1]:.2%}")


## 4. Comparing a Few Different Passenger Profiles

A single prediction doesn't show much about *how* the model behaves. Running a handful of contrasting profiles through it side by side — varying class, sex, and age, the historically most influential factors — makes the model's behavior much easier to see and sanity-check.

In [ ]:
import pandas as pd

profiles = pd.DataFrame([
    [1, 'female', 5.0,  0, 1, 100.0, 'S'],   # 1st class young girl
    [1, 'male',   40.0, 1, 0, 80.0,  'C'],   # 1st class adult man
    [3, 'female', 22.0, 0, 0, 7.5,   'S'],   # 3rd class young woman
    [3, 'male',   25.0, 0, 0, 7.5,   'S'],   # 3rd class young man
], columns=['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked'])

profiles['survival_probability'] = pipe.predict_proba(profiles.values)[:, 1]
profiles


### Visualizing the Comparison

In [ ]:
import matplotlib.pyplot as plt

labels = ['1st class girl', '1st class man', '3rd class woman', '3rd class man']
plt.bar(labels, profiles['survival_probability'], color=['#4C72B0', '#4C72B0', '#DD8452', '#DD8452'])
plt.ylabel('Predicted probability of survival')
plt.title('Model behavior across passenger profiles')
plt.ylim(0, 1)
plt.show()


## Summary

| Task | Method |
|---|---|
| Load a saved pipeline | `pickle.load(open('pipe.pkl', 'rb'))` |
| Predict on raw, unprocessed input | `pipe.predict(raw_array)` — no manual preprocessing needed |
| Get prediction confidence | `pipe.predict_proba(raw_array)` |
| Compare multiple inputs at once | pass a 2D array/DataFrame with several rows |

Compare this to `predict-without-pipeline.ipynb`: same end result, but there, every preprocessing step has to be repeated **by hand**, in the exact right order, using three separately-loaded `.pkl` files — one pipeline object here replaces all of that.